# Bias Audit: Hiring Model

Interactive walkthrough of the bias detection pillar.

**What we cover:**
1. Load hiring prediction data with intentional bias
2. Demographic parity testing (four-fifths rule)
3. Equalized odds analysis
4. Intersectional bias detection (gender x ethnicity)
5. Visualize disparities with heatmaps and bar charts

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.figsize'] = (10, 5)

from src.bias_detection.demographic_parity import DemographicParityTester
from src.bias_detection.equalized_odds import EqualizedOddsTester
from src.bias_detection.intersectional import IntersectionalAnalyzer

## 1. Load Hiring Predictions

The synthetic dataset contains an intentional 5% bias against one ethnicity group and a subtle age bias for the 51-60 group.

In [ ]:
hiring = pd.read_csv('../data/hiring_predictions.csv')
print(f'Candidates: {len(hiring):,}')
print(f'Hire rate: {hiring["model_prediction"].mean():.1%}')
print(f'\nDemographic breakdown:')
for col in ['gender', 'age_group', 'ethnicity']:
    print(f'  {col}: {dict(hiring[col].value_counts())}')
hiring.head()

## 2. Demographic Parity (Four-Fifths Rule)

The EEOC's four-fifths rule: the selection rate for any group must be at least 80% of the rate for the group with the highest rate.

In [ ]:
tester = DemographicParityTester(threshold=0.8)
protected = ['gender', 'age_group', 'ethnicity']

# Disparate impact summary
di_summary = tester.disparate_impact_summary(hiring, 'model_prediction', protected)
display(di_summary)

In [ ]:
# Positive rate by group
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, attr in zip(axes, protected):
    rates = hiring.groupby(attr)['model_prediction'].mean().sort_values()
    colors = ['#e74c3c' if r / rates.max() < 0.8 else '#2ecc71' for r in rates]
    rates.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(f'Hire Rate by {attr.replace("_", " ").title()}')
    ax.set_xlabel('Positive Prediction Rate')
    ax.axvline(rates.max() * 0.8, color='red', linestyle='--', alpha=0.7, label='80% threshold')
    ax.legend(fontsize=7)

plt.suptitle('Demographic Parity Analysis (red = fails four-fifths rule)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed group results for ethnicity (the failing attribute)
eth_result = tester.test(hiring, 'model_prediction', 'ethnicity')
print(f'Ethnicity Disparate Impact Ratio: {eth_result["disparate_impact_ratio"]:.3f}')
print(f'Passes Four-Fifths Rule: {eth_result["passes_four_fifths_rule"]}')
print(f'\nGroup-level rates:')
for gr in eth_result['group_results']:
    flag = ' *** BELOW THRESHOLD' if not gr.passes_threshold else ''
    print(f'  {gr.group}: {gr.positive_rate:.3f} (n={gr.count}){flag}')

## 3. Equalized Odds Analysis

In [ ]:
odds_tester = EqualizedOddsTester(max_tpr_gap=0.1, max_fpr_gap=0.1)
# Using model_prediction as both prediction and ground truth (since these are model outputs)
odds_results = odds_tester.test_multiple(hiring, 'model_prediction', 'model_prediction', protected)

gaps = odds_tester.gap_summary(hiring, 'model_prediction', 'model_prediction', protected)
display(gaps)

In [ ]:
# Detailed rates table
rates_table = odds_tester.summary_table(hiring, 'model_prediction', 'model_prediction', protected)
display(rates_table)

## 4. Intersectional Bias Analysis

Single-attribute analysis may miss compound disadvantages. For example, the model might be fair for gender AND ethnicity independently, but biased against a specific intersection (e.g., older women of a particular ethnicity).

In [ ]:
analyzer = IntersectionalAnalyzer(min_group_size=30)
report = analyzer.disparity_report(hiring, 'model_prediction', protected)
print(f'Intersectional groups analyzed: {len(report)}')
print(f'\nMost disadvantaged groups:')
display(report.head(10))
print(f'\nMost advantaged groups:')
display(report.tail(5))

In [ ]:
# Intersectional heatmap: gender x ethnicity
heatmap = analyzer.heatmap_data(hiring, 'model_prediction', 'ethnicity', 'gender')
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(heatmap.values, cmap='RdYlGn', aspect='auto', vmin=0.5, vmax=0.85)
ax.set_xticks(range(len(heatmap.columns)))
ax.set_xticklabels(heatmap.columns)
ax.set_yticks(range(len(heatmap.index)))
ax.set_yticklabels(heatmap.index)
# Annotate
for i in range(len(heatmap.index)):
    for j in range(len(heatmap.columns)):
        val = heatmap.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=10,
                   fontweight='bold', color='white' if val < 0.6 else 'black')
plt.colorbar(im, label='Hire Rate')
ax.set_title('Intersectional Hire Rates: Ethnicity x Gender')
ax.set_xlabel('Gender')
ax.set_ylabel('Ethnicity')
plt.tight_layout()
plt.show()

In [ ]:
# Ethnicity x Age Group heatmap
heatmap2 = analyzer.heatmap_data(hiring, 'model_prediction', 'ethnicity', 'age_group')
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(heatmap2.values, cmap='RdYlGn', aspect='auto', vmin=0.5, vmax=0.85)
ax.set_xticks(range(len(heatmap2.columns)))
ax.set_xticklabels(heatmap2.columns)
ax.set_yticks(range(len(heatmap2.index)))
ax.set_yticklabels(heatmap2.index)
for i in range(len(heatmap2.index)):
    for j in range(len(heatmap2.columns)):
        val = heatmap2.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=10,
                   fontweight='bold', color='white' if val < 0.6 else 'black')
plt.colorbar(im, label='Hire Rate')
ax.set_title('Intersectional Hire Rates: Ethnicity x Age Group')
ax.set_xlabel('Age Group')
ax.set_ylabel('Ethnicity')
plt.tight_layout()
plt.show()

## Key Findings

- **Ethnicity bias detected**: Disparate impact ratio of 0.732 fails the four-fifths rule
- **Age bias present but borderline**: 51-60 age group shows lower rates, DI ratio at 0.817 (passes but close)
- **Intersectional compounds**: The most disadvantaged group (specific ethnicity x age combination) faces a ~9% lower hire rate than the overall average
- **Remediation needed**: Model should not be deployed until ethnicity bias is addressed through feature review, retraining, or post-processing calibration